# Homework: Predicting Marine Fish Life-History Strategies with a Multilayer Perceptron

## Overview

Marine fish communities differ in their life-history strategies. Some communities are dominated by **fast-living species** that tend to grow quickly and mature earlier, while others are dominated by **slow-living species** that tend to mature later and live longer.

In this homework, you will use environmental conditions to predict whether a marine fish community is classified as:

- `0` = **slow-life-history community**
- `1` = **fast-life-history community**

You will build a **multilayer perceptron (MLP)** with TensorFlow/Keras and compare it with a logistic-regression baseline.

### Learning objectives

By the end of this assignment, you should be able to:

1. Prepare tabular environmental data for a neural network.
2. Explain why feature scaling matters for MLPs.
3. Build and train an MLP for binary classification.
4. Use a validation set rather than the test set to make modeling decisions.
5. Diagnose underfitting and overfitting from training history.
6. Apply early stopping and regularization.
7. Compare an MLP with a simpler linear model.
8. Interpret model performance in an ecological context.

### Variables in the student dataset

The file provided with this assignment is:

`FishTraits_MLP_Student.csv`

It contains the following columns:

| Variable | Description |
|---|---|
| `site_id` | Unique sampling/grid-cell identifier |
| `longitude`, `latitude` | Site coordinates |
| `Depth` | Water depth |
| `SBT` | Mean sea-bottom temperature |
| `SBS` | Mean sea-bottom salinity |
| `Chl` | Mean chlorophyll-a concentration |
| `SBT_var` | Seasonal variability in bottom temperature |
| `Chl_var` | Seasonal variability in chlorophyll-a |
| `Fishing` | Fishing-pressure indicator |
| `life_history_class` | Response class: `slow` or `fast` |
| `split` | Instructor-provided data split: `train`, `validation`, or `test` |
| `spatial_block` | Spatial-block identifier reserved for later spatial-validation exercises |

### Important modeling rule

Use only these seven environmental predictors:

```text
Depth
SBT
SBS
Chl
SBT_var
Chl_var
Fishing
```

Do **not** use `site_id`, `longitude`, `latitude`, `split`, or `spatial_block` as predictors.

The response variable is `life_history_class`.

For modeling, encode:

- `slow` = 0
- `fast` = 1

The dataset already contains instructor-defined **train, validation, and test splits**. Use the `split` column rather than making a new random split.

## Submission

Submit your completed notebook with:

- all code cells executed,
- all requested figures,
- written answers to every question,
- and a short final interpretation.

Your written answers should explain **what the model is doing and what the results mean**, not simply report numbers.

## 1. Import packages and confirm TensorFlow works

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
)

import tensorflow as tf

## 2. Load and inspect the data

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/FishTraits.csv"

fish = pd.read_csv(DATA_URL)

print("Shape:", fish.shape)
fish.head()

In [ ]:
fish.info()

In [ ]:
fish["life_history_class"].value_counts()

In [ ]:
fish["life_history_class"].value_counts(normalize=True)

### Question 1

1. How many observations are in the dataset?
2. Is the target approximately balanced?
3. Why would severe class imbalance make raw accuracy potentially misleading?

## 3. Explore environmental differences between classes

In [ ]:
predictors = [
    "Depth",
    "SBT",
    "SBS",
    "Chl",
    "SBT_var",
    "Chl_var",
    "Fishing",
]

fish.groupby("life_history_class")[predictors].mean().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

variables_to_plot = [
    "SBT",
    "Depth",
    "SBT_var",
    "Chl",
]

for ax, variable in zip(axes.ravel(), variables_to_plot):
    for target_value, label in [("slow", "slow"), ("fast", "fast")]:
        values = fish.loc[
            fish["life_history_class"] == target_value,
            variable
        ].dropna()

        ax.hist(
            values,
            bins=25,
            alpha=0.5,
            density=True,
            label=label,
        )

    ax.set_title(variable)
    ax.set_ylabel("Density")
    ax.legend()

plt.tight_layout()
plt.show()

### Question 2

Describe at least **two environmental variables** that appear to differ between fast- and slow-life-history communities.

Do these patterns prove that the environmental variable *causes* the life-history strategy? Explain.

## 4. Define predictors and target

In [ ]:
X = fish[predictors].copy()

y = (
    fish["life_history_class"]
    .map({"slow": 0, "fast": 1})
    .astype(int)
)

print(X.shape)
print(y.shape)
print(y.value_counts())

## 5. Use the instructor-provided training, validation, and test sets

The CSV already contains a `split` column. Use it exactly as provided:

- `train` for model fitting,
- `validation` for model selection and early stopping,
- `test` for final evaluation only.

This keeps the class exercise reproducible and ensures that every student evaluates the same observations.

Use:

- **60%** of the observations for model fitting,
- **20%** for validation,
- **20%** for the final test set.

The test set must remain untouched while you compare model settings.

In [ ]:
train_mask = fish["split"].eq("train")
validation_mask = fish["split"].eq("validation")
test_mask = fish["split"].eq("test")

X_fit = fish.loc[train_mask, predictors].copy()
X_validation = fish.loc[validation_mask, predictors].copy()
X_test = fish.loc[test_mask, predictors].copy()

y_fit = (
    fish.loc[train_mask, "life_history_class"]
    .map({"slow": 0, "fast": 1})
    .astype(int)
)

y_validation = (
    fish.loc[validation_mask, "life_history_class"]
    .map({"slow": 0, "fast": 1})
    .astype(int)
)

y_test = (
    fish.loc[test_mask, "life_history_class"]
    .map({"slow": 0, "fast": 1})
    .astype(int)
)

print("Fit:", X_fit.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

print("\nClass counts by split:")
print(
    fish.groupby(["split", "life_history_class"])
        .size()
        .unstack(fill_value=0)
)

## 6. Scale the environmental predictors

In [ ]:
scaler = StandardScaler()

X_fit_scaled = scaler.fit_transform(X_fit)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

## 7. Establish a logistic-regression baseline

In [ ]:
logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42,
)

logistic_model.fit(X_fit_scaled, y_fit)

logistic_validation_probability = logistic_model.predict_proba(
    X_validation_scaled
)[:, 1]

logistic_validation_pred = (
    logistic_validation_probability >= 0.5
).astype(int)

print(
    "Validation accuracy:",
    accuracy_score(y_validation, logistic_validation_pred),
)

print(
    "Validation balanced accuracy:",
    balanced_accuracy_score(y_validation, logistic_validation_pred),
)

print(
    "Validation ROC AUC:",
    roc_auc_score(y_validation, logistic_validation_probability),
)

### Question 3

Why is logistic regression a useful baseline before fitting an MLP?

What kind of decision relationship can an MLP represent that logistic regression cannot represent without manually creating nonlinear features?

## 8. Build your first MLP

Build a neural network with:

- 7 environmental inputs,
- one hidden layer with **8 units**,
- ReLU activation in the hidden layer,
- one sigmoid output unit,
- Adam optimizer,
- binary cross-entropy loss.

Complete the missing lines.

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(predictors),)),

    # TODO: add a Dense hidden layer with 8 units and ReLU activation


    # TODO: add the binary-classification output layer

])

model.compile(
    # TODO: choose the Adam optimizer
    optimizer=____________________,

    # TODO: choose the appropriate binary classification loss
    loss=____________________,

    metrics=["accuracy"],
)

model.summary()

## 9. Train the MLP

In [ ]:
history = model.fit(
    X_fit_scaled,
    y_fit,
    validation_data=(X_validation_scaled, y_validation),
    epochs=200,
    batch_size=32,
    verbose=0,
)

## 10. Visualize the learning history

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df.index + 1, history_df["loss"], label="training")
axes[0].plot(history_df.index + 1, history_df["val_loss"], label="validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history_df.index + 1, history_df["accuracy"], label="training")
axes[1].plot(
    history_df.index + 1,
    history_df["val_accuracy"],
    label="validation",
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

### Question 4

Examine both plots.

1. Does the model appear to be underfitting, overfitting, or reasonably fit?
2. Around what epoch does validation loss stop improving substantially?
3. Why can validation accuracy remain nearly constant while validation loss continues to change?

## 11. Add early stopping

Now train the same model with early stopping.

Monitor **validation loss**, use:

- `patience=20`
- `min_delta=1e-4`
- `restore_best_weights=True`
- maximum of 1000 epochs

In [ ]:
tf.keras.utils.set_random_seed(42)

early_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(predictors),)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

early_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=______,#TODO
    min_delta=______,#TODO
    restore_best_weights=______,#TODO
)

early_history = early_model.fit(
    X_fit_scaled,
    y_fit,
    validation_data=(X_validation_scaled, y_validation),
    epochs=______,#TODO
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0,
)

print("Epochs actually trained:", len(early_history.history["loss"]))

In [ ]:
early_history_df = pd.DataFrame(early_history.history)

best_epoch = early_history_df["val_loss"].idxmin() + 1
stopped_epoch = len(early_history_df)

plt.figure(figsize=(8, 5))

plt.plot(
    early_history_df.index + 1,
    early_history_df["val_loss"],
    label="validation loss",
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label="best validation loss",
)

plt.axvline(
    stopped_epoch,
    linestyle=":",
    label="training stopped",
)

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Early stopping based on validation loss")
plt.legend()
plt.show()

### Question 5

1. At what epoch was validation loss lowest?
2. At what epoch did training stop?
3. Why are those two epochs not necessarily identical?
4. What does `restore_best_weights=True` do?

## 12. Experiment with model complexity

In [ ]:
def build_mlp(
    hidden_units=8,
    dropout_rate=0.0,
    l2_strength=0.0,
):
    regularizer = (
        tf.keras.regularizers.l2(l2_strength)
        if l2_strength > 0
        else None
    )

    layers = [
        tf.keras.layers.Input(shape=(len(predictors),)),
        tf.keras.layers.Dense(
            hidden_units,
            activation="relu",
            kernel_regularizer=regularizer,
        ),
    ]

    if dropout_rate > 0:
        layers.append(
            tf.keras.layers.Dropout(dropout_rate)
        )

    layers.append(
        tf.keras.layers.Dense(1, activation="sigmoid")
    )

    model = tf.keras.Sequential(layers)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model

In [ ]:
def evaluate_setting(
    label,
    hidden_units=8,
    dropout_rate=0.0,
    l2_strength=0.0,
):
    tf.keras.utils.set_random_seed(42)

    model = build_mlp(
        hidden_units=hidden_units,
        dropout_rate=dropout_rate,
        l2_strength=l2_strength,
    )

    callback = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        min_delta=1e-4,
        restore_best_weights=True,
    )

    history = model.fit(
        X_fit_scaled,
        y_fit,
        validation_data=(X_validation_scaled, y_validation),
        epochs=1000,
        batch_size=32,
        callbacks=[callback],
        verbose=0,
    )

    probability = model.predict(
        X_validation_scaled,
        verbose=0,
    ).ravel()

    pred = (probability >= 0.5).astype(int)

    return {
        "setting": label,
        "hidden_units": hidden_units,
        "dropout": dropout_rate,
        "l2": l2_strength,
        "epochs_used": len(history.history["loss"]),
        "validation_accuracy": accuracy_score(
            y_validation,
            pred,
        ),
        "validation_balanced_accuracy": balanced_accuracy_score(
            y_validation,
            pred,
        ),
        "validation_auc": roc_auc_score(
            y_validation,
            probability,
        ),
        "best_validation_loss": np.min(
            history.history["val_loss"]
        ),
    }

Change **one setting at a time**. For each experiment, train using the same fit and validation data and record the validation results.

Complete at least these experiments:

1. `hidden_units=1`
2. `hidden_units=32`
3. `hidden_units=64`
4. baseline 8-unit model + dropout of `0.30`
5. baseline 8-unit model + L2 regularization

Use early stopping for each experiment.

In [ ]:
experiments = [
    # TODO: add the required experimental settings here.
    # Example:
    # evaluate_setting("1 hidden unit", hidden_units=1),
]
# Variables to change are hidden_units, dropout_rate, and l2_strength

results = pd.DataFrame(experiments)
results

### Question 6

Using only the **validation results**:

1. Does the 1-unit model appear to underfit?
2. Does increasing the number of hidden units consistently improve validation performance?
3. Did dropout help?
4. Did L2 regularization help?
5. Which model setting would you carry forward for final evaluation, and why?

## 13. Final model evaluation

Choose the model setting you selected using validation performance.

Refit that model using the same fit/validation workflow and early stopping. Then evaluate it **once** on the test set.

In [ ]:
# TODO: replace these values with the settings you selected.
FINAL_HIDDEN_UNITS = ______
FINAL_DROPOUT = ______
FINAL_L2 = ______

tf.keras.utils.set_random_seed(42)

final_model = build_mlp(
    hidden_units=FINAL_HIDDEN_UNITS,
    dropout_rate=FINAL_DROPOUT,
    l2_strength=FINAL_L2,
)

final_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    min_delta=1e-4,
    restore_best_weights=True,
)

final_history = final_model.fit(
    X_fit_scaled,
    y_fit,
    validation_data=(X_validation_scaled, y_validation),
    epochs=1000,
    batch_size=32,
    callbacks=[final_callback],
    verbose=0,
)

test_probability = final_model.predict(
    X_test_scaled,
    verbose=0,
).ravel()

test_pred = (
    test_probability >= 0.5
).astype(int)

print(
    "Test accuracy:",
    accuracy_score(y_test, test_pred),
)

print(
    "Test balanced accuracy:",
    balanced_accuracy_score(y_test, test_pred),
)

print(
    "Test ROC AUC:",
    roc_auc_score(y_test, test_probability),
)

print()
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["slow", "fast"],
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    display_labels=["slow", "fast"],
)

plt.title("Final MLP: test-set confusion matrix")
plt.show()

## 14. Compare the MLP with logistic regression on the test set

Only now should you evaluate the logistic-regression baseline on the test set.

In [ ]:
logistic_test_probability = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

logistic_test_pred = (
    logistic_test_probability >= 0.5
).astype(int)

comparison = pd.DataFrame({
    "model": [
        "Logistic regression",
        "MLP",
    ],
    "test_accuracy": [
        accuracy_score(
            y_test,
            logistic_test_pred,
        ),
        accuracy_score(
            y_test,
            test_pred,
        ),
    ],
    "test_balanced_accuracy": [
        balanced_accuracy_score(
            y_test,
            logistic_test_pred,
        ),
        balanced_accuracy_score(
            y_test,
            test_pred,
        ),
    ],
    "test_roc_auc": [
        roc_auc_score(
            y_test,
            logistic_test_probability,
        ),
        roc_auc_score(
            y_test,
            test_probability,
        ),
    ],
})

comparison

### Question 7

Did the MLP clearly outperform logistic regression?

Your answer should consider:

- accuracy,
- balanced accuracy,
- ROC AUC,
- model complexity,
- and whether the improvement is large enough to justify the more complicated model.

A neural network is not automatically the better scientific model simply because it is more flexible.

# Grading guide

| Component | Points |
|---|---:|
| Data inspection | 1 |
| Train/validation/test design and scaling | 1 |
| Logistic-regression baseline | 1 |
| Correct MLP construction and training | 2 |
| Learning curves and early stopping | 4 |
| Model-complexity / regularization experiments | 4 |
| Final test evaluation and baseline comparison | 2 |
| Questions 1 - 7 | 10 |
| **Total** | **25** |